In [1]:
# Imports and Configurations
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import time

# Clustering and distance metrics
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score

# Parallelization for OCP
import multiprocessing as mp

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)

# Configuration
BENCHMARK        = '^GSPC'
RISK_FREE_RATE   = 0.045
START_DATE       = '2014-01-01'
END_DATE         = '2024-01-01'
ROLLING_WINDOW   = 52
MIN_HISTORY      = 400
K_VALUES         = [2,4,7]
N_CORES          = 14

# Three-period out-of-sample validation framework
TRAIN_START = '2014-01-01'
TRAIN_END   = '2017-12-31'
VAL_START   = '2018-01-01'
VAL_END     = '2020-12-31'
TEST_START  = '2021-01-01'
TEST_END    = '2023-12-31'

# Strategy Parameters
ZSCORE_WINDOW   = 52
VOL_WINDOW      = 26
VOL_THRESHOLD   = 1.5
HIGH_VOL_ENTRY  = 2.5
LOW_VOL_ENTRY   = 1.5
STANDARD_ENTRY  = 2.0
EXIT_THRESHOLD  = 0.5
STOP_LOSS       = 3.5

print('Notebook 6: DTW vs. OCP Clustering - S&P 500 Universe')
print(f'Period: {START_DATE} to {END_DATE}')
print(f'Risk-free rate: {RISK_FREE_RATE} (applied to both DTW and OCP Sharpe inputs)')
print(f'Parallelization: {N_CORES} cores for OCP')
print(f'\nThree-period framework (for later validation stage):')
print(f'    Train: {TRAIN_START} to {TRAIN_END}')
print(f'    Val:   {VAL_START} to {VAL_END}')
print(f'    Test:  {TEST_START} to {TEST_END}')
print(f'\nStrategy parameters (for later backtest stage):')
print(f'    Entry: ±{HIGH_VOL_ENTRY}σ (high vol) / ±{LOW_VOL_ENTRY}σ (low vol)')
print(f'    Exit:  ±{EXIT_THRESHOLD}σ')
print(f'    Stop:  ±{STOP_LOSS}σ')

Notebook 6: DTW vs. OCP Clustering - S&P 500 Universe
Period: 2014-01-01 to 2024-01-01
Risk-free rate: 0.045 (applied to both DTW and OCP Sharpe inputs)
Parallelization: 14 cores for OCP

Three-period framework (for later validation stage):
    Train: 2014-01-01 to 2017-12-31
    Val:   2018-01-01 to 2020-12-31
    Test:  2021-01-01 to 2023-12-31

Strategy parameters (for later backtest stage):
    Entry: ±2.5σ (high vol) / ±1.5σ (low vol)
    Exit:  ±0.5σ
    Stop:  ±3.5σ


In [7]:
# Retrieving S&P500 constituents as of January 1, 2014

# This source avoids survivorship bias
hist_url = (
    'https://raw.githubusercontent.com/fja05680/sp500/master/'
    'S%26P%20500%20Historical%20Components%20%26%20Changes.csv'
)

hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])

# Finding the row closest to our target date
target_date = pd.Timestamp('2014-01-01')
snapshot_row = hist[hist['date'] <= target_date].sort_values('date').iloc[-1]

print(f'Using snapshot date: {snapshot_row['date'].date()}')

sp500_tickers_2014 = sorted(snapshot_row['tickers'].split(','))

sp500_tickers_2014 = [t.strip().replace('.','-') for t in sp500_tickers_2014]

print(f'S&P 500 constituents as of {target_date.date()}: {len(sp500_tickers_2014)} stocks')
print(f'\nFirst 10 tickers: {sp500_tickers_2014[:10]}')

Using snapshot date: 2013-12-31
S&P 500 constituents as of 2014-01-01: 502 stocks

First 10 tickers: ['A', 'AABA', 'AAPL', 'ABBV', 'ABC', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM']


In [9]:
# Downloading price date for the S&P 500 2014 universe
import yfinance as yf

print(f'Downloading price data for {len(sp500_tickers_2014)} stocks..')
print(f'Period: {START_DATE} to {END_DATE}, weekly (Wednesday), adjusted close\n')

raw = yf.download(
    sp500_tickers_2014,
    start=START_DATE,
    end=END_DATE,
    interval='1wk',
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_date = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')

# Dropping any ticker without complete history across the full window 
before_drop = price_data.shape[1]
price_data  = price_data.ffill().dropna(axis=1)
after_drop  = price_data.shape[1]

print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Tickers with complete history: {after_drop} (dropped {before_drop - after_drop} incomplete/delisted)')

dropped_tickers = sorted(set(sp500_tickers_2014) - set(price_data.columns))
print(f'\nDropped tickers ({len(dropped_tickers)}): {dropped_tickers}')

Period: 2014-01-01 to 2024-01-01, weekly (Wednesday), adjusted close



$SWY-201501: possibly delisted; no timezone found
$UTX: possibly delisted; no timezone found       ]
$DFS: possibly delisted; no timezone found
[                       0%                       ]  2 of 502 completed$DNB: possibly delisted; no timezone found
[*                      2%                       ]  8 of 502 completed$SNDK-201605: possibly delisted; no timezone found
$CERN: possibly delisted; no timezone found      ]  13 of 502 completed
$GPS: possibly delisted; no timezone found       ]  17 of 502 completed
$RDC: possibly delisted; no timezone found       ]  18 of 502 completed
$BRCM-201601: possibly delisted; no timezone found  19 of 502 completed
$CMA: possibly delisted; no timezone found       ]  19 of 502 completed
[**                     5%                       ]  23 of 502 completed$WIN: possibly delisted; no timezone found
$CHK: possibly delisted; no timezone found       ]  25 of 502 completed
[**                     5%                       ]  26 of 502 completed$AABA


Price data shape: (964, 361)
Date range: 2014-01-01 to 2023-12-27
Tickers with complete history: 361 (dropped 141 incomplete/delisted)

Dropped tickers (141): ['AABA', 'ABC', 'ADS', 'ADT-201604', 'AGN', 'AGN-201503', 'ALTR-201512', 'ALXN', 'ANTM', 'APC', 'ARG-201605', 'ARNC', 'AVP', 'BCR-201712', 'BEAM-201404', 'BHGE', 'BK', 'BLL', 'BRCM-201601', 'BTUUQ-201704', 'CA', 'CAM-201604', 'CB-201601', 'CBS', 'CCE', 'CELG', 'CERN', 'CFN-201503', 'CHK', 'CMA', 'COG', 'COV-201501', 'CTL', 'CTXS', 'CVC-201606', 'DD-201708', 'DFS', 'DISCA', 'DNB', 'DNR', 'DO', 'DOW-201708', 'DTV-201507', 'EMC-201609', 'ESV', 'ETFC', 'FB', 'FDO-201507', 'FLIR', 'FOXA', 'FRX-201406', 'FTI-201701', 'FTR', 'GAS-201606', 'GGP', 'GGP-201808', 'GPS', 'HAR-201703', 'HCBK-201510', 'HCP', 'HES', 'HOT-201609', 'HRS', 'HSP-201509', 'IGT-201504', 'IPG', 'IR', 'JCI-201609', 'JEC', 'JNPR', 'JOY-201704', 'JWN', 'K', 'KORS', 'KRFT-201507', 'KSU', 'LB', 'LIFE-201402', 'LLL', 'LLTC-201703', 'LM', 'LO-201506', 'LSI-201405', 'MJN-201